<a href="https://colab.research.google.com/github/MishaE-e/ml_fmi/blob/main/8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install dm-haiku optax chex gymnasium

In [2]:
import copy
from shutil import rmtree
import random
import collections
import numpy as np
import gym
from gym.wrappers import RecordVideo
import jax
import jax.numpy as jnp
import haiku as hk
import optax
import matplotlib.pyplot as plt
from IPython.display import HTML
from base64 import b64encode
import chex
import warnings
warnings.filterwarnings('ignore')
import gymnasium as gym
from collections import namedtuple

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


1.

In [3]:
def linear_policy(params, obs):
    dot_product_result = jnp.dot(params, obs)

    action = jax.lax.select(
        dot_product_result > 0,
        1,
        0,
    )
    return action

проверка

In [4]:
fixed_obs = jnp.array([1.0, 1.0, 2.0, 4.0])
params1 = jnp.array([1.0, 1.0, 1.0, 1.0])
params2 = jnp.array([-1.0, -1.0, -1.0, -1.0])
res1 = linear_policy(params1, fixed_obs)
res2 = linear_policy(params2, fixed_obs)

print(f"{res1}, {res2}")

1, 0


2.

In [5]:
def run_episode(env):
    episode_return = 0
    done = False
    params = jnp.array([1, -2, 2, -1])

    obs, _ = env.reset()

    while not done:
        action = linear_policy(params, obs)
        action = np.array(action)

        obs, reward, done, truncated, info = env.step(action)
        episode_return += reward
        done = done or truncated

    return episode_return

проверка

In [6]:
test_env = gym.make("CartPole-v1")
result = run_episode(test_env)
print(f"{result}")
test_env.close()

20.0


3.

In [7]:
RandomPolicySearchParams = namedtuple("RandomPolicySearchParams", ["current", "best"])
def random_policy_search_choose_action(
    key,
    params,
    actor_state,
    obs,
    evaluation=False
):
    best_action = linear_policy(params.best, obs)
    current_action = linear_policy(params.current, obs)

    action = jax.lax.select(
        evaluation,
        best_action,
        current_action
    )

    return action, actor_state

проверка

In [8]:
obs = np.ones(4)
current_params = np.ones(4) * -1
best_params = np.ones(4)
rps_params = RandomPolicySearchParams(current_params, best_params)
action1, _ = random_policy_search_choose_action(None, rps_params, None, obs, evaluation=False)
action2, _ = random_policy_search_choose_action(None, rps_params, None, obs, evaluation=True)
print(f"Действие без оценки: {action1}")
print(f"Действия с оценкой: {action2}")

Действие без оценки: 0
Действия с оценкой: 1


4.

In [9]:
def get_new_random_weights(random_key, old_weights, minval=-2.0, maxval=2.0):
    new_weights_shape = old_weights.shape
    new_weights_dtype = old_weights.dtype

    new_params = jax.random.uniform(
        random_key,
        shape=new_weights_shape,
        dtype=new_weights_dtype,
        minval=minval,
        maxval=maxval
    )

    return new_params

проверка

In [10]:
old_weights = np.ones(4, dtype=np.float32)
random_key = jax.random.PRNGKey(42)
new_weights = get_new_random_weights(random_key, old_weights, -2.0, 2.0)
print(f"{new_weights}")

[-0.04516172  0.7191887   0.46508598  0.24406433]


5.

In [11]:
RandomPolicyLearnState = namedtuple("RandomPolicyLearnState", ["best_average_episode_return"])
def random_policy_search_learn(key, params, learn_state, memory):
    best_params = params.best
    current_params = params.current

    current_average_episode_return = memory
    best_average_episode_return = learn_state.best_average_episode_return

    best_params = jax.lax.select(
        current_average_episode_return > best_average_episode_return,
        current_params,
        best_params
    )

    best_average_episode_return = jax.lax.select(
        current_average_episode_return > best_average_episode_return,
        current_average_episode_return,
        best_average_episode_return
    )

    new_params = get_new_random_weights(key, current_params)

    params = RandomPolicySearchParams(current=new_params, best=best_params)

    return params, RandomPolicyLearnState(best_average_episode_return)

проверка

In [12]:
params = RandomPolicySearchParams(
    np.ones(4, dtype=np.float32),
    np.ones(4, dtype=np.float32) * -1
)
learn_state = RandomPolicyLearnState(10)
memory = 11
key = jax.random.PRNGKey(42)
new_params, new_learn_state = random_policy_search_learn(key, params, learn_state, memory)
print(f"Текущие параметры: {new_params.current}")
print(f"Лучшие параметры: {new_params.best}")
print(f"Лучшая средняя отдача эпизода: {new_learn_state.best_average_episode_return}")

Текущие параметры: [-0.04516172  0.7191887   0.46508598  0.24406433]
Лучшие параметры: [1. 1. 1. 1.]
Лучшая средняя отдача эпизода: 11


6.

In [13]:
def compute_weighted_log_prob(action_prob, episode_return):
    log_prob = jnp.log(action_prob)
    weighted_log_prob = log_prob * episode_return
    return weighted_log_prob

проверка

In [14]:
result = compute_weighted_log_prob(0.8, 100)
print(f"{result}")


-22.314353942871094


7.

In [15]:
def compute_rewards_to_go(rewards):
    rewards_to_go = []
    total = 0
    for r in reversed(rewards):
        total += r
        rewards_to_go.append(total)
    rewards_to_go.reverse()
    return rewards_to_go

проверка

In [16]:
result = compute_rewards_to_go([1, 2, 3, 4])
print(f"Rewards: [1, 2, 3, 4]")
print(f"Rewards-to-go: {result}")


Rewards: [1, 2, 3, 4]
Rewards-to-go: [10, 9, 7, 4]


8.

In [17]:
def sample_action(random_key, logits):
    probs = jax.nn.softmax(logits)
    action = jax.random.categorical(random_key, logits)
    return action

проверка

In [18]:
random_key = jax.random.PRNGKey(42)
logits = np.array([1.0, 2.0])
action = sample_action(random_key, logits)
print(f"Logits: {logits}")
print(f"Action: {action}")

Logits: [1. 2.]
Action: 1


9.

In [19]:
def policy_gradient_loss(action, logits, reward_to_go):
    all_action_probs = jax.nn.softmax(logits)
    action_prob = all_action_probs[action]
    weighted_log_prob = jnp.log(action_prob) * reward_to_go
    loss = -weighted_log_prob
    return loss

проверка

In [20]:
result = policy_gradient_loss(1, np.array([1.0, 2.0]), 10)
print(f"Потери: {result}")

Потери: 3.1326165199279785


10.

In [21]:
def select_greedy_action(q_values):
    action = jnp.argmax(q_values)
    return action

проверка

In [22]:
q_values = jnp.array([1.0, 1.0, 3.0, 4.0])
action = select_greedy_action(q_values)
print(f"Q-values: {q_values}")
print(f"Greedy action: {action}")

Q-values: [1. 1. 3. 4.]
Greedy action: 3


11.

In [23]:
def compute_squared_error(pred, target):
    squared_error = jnp.square(pred - target)
    return squared_error


проверка

In [24]:
result = compute_squared_error(1.0, 4.0)
print(f"Pred=1, Target=4, Squared error={result}")

Pred=1, Target=4, Squared error=9.0


12.

In [25]:
def compute_bellman_target(reward, done, next_q_values):
    bellman_target = reward + (1.0 - done) * jnp.max(next_q_values)
    return bellman_target

проверка

In [26]:
res1 = compute_bellman_target(1.0, 0.0, np.array([3.0, 2.0]))
res2 = compute_bellman_target(1.0, 1.0, np.array([3.0, 2.0]))
print(f"Not done (should be 1 + max(3,2)=4): {res1}")
print(f"Done (should be just reward=1): {res2}")


Not done (should be 1 + max(3,2)=4): 4.0
Done (should be just reward=1): 1.0


13.

In [27]:
def q_learning_loss(q_values, action, reward, done, next_q_values):
    chosen_action_q_value = q_values[action]  # q_value of action
    bellman_target = compute_bellman_target(reward, done, next_q_values)
    squared_error = compute_squared_error(chosen_action_q_value, bellman_target)
    return squared_error

проверка

In [28]:
result = q_learning_loss(np.array([3.0, 2.0]), 1, 2.0, 0.0, np.array([3.0, 2.0]))
print(f"Q-loss: {result}")


Q-loss: 9.0


14.

In [29]:
def select_random_action(key, num_actions):
    action = jax.random.randint(key, shape=(), minval=0, maxval=num_actions)
    return action

проверка

In [30]:
key1 = jax.random.PRNGKey(6)
key2 = jax.random.PRNGKey(1000)
res1 = select_random_action(key1, 2)
res2 = select_random_action(key2, 2)
print(f"Random action with key=6: {res1}")
print(f"Random action with key=1000: {res2}")

Random action with key=6: 0
Random action with key=1000: 0


15.

In [31]:
EPSILON_DECAY_TIMESTEPS = 3000
EPSILON_MIN = 0.1

def get_epsilon(num_timesteps):
    epsilon = 1.0 - (num_timesteps / EPSILON_DECAY_TIMESTEPS) * (1.0 - EPSILON_MIN)

    epsilon = jax.lax.select(
        epsilon < EPSILON_MIN,
        EPSILON_MIN,
        epsilon
    )
    return epsilon

проверка

In [32]:
res1 = get_epsilon(10)
res2 = get_epsilon(3000)
res3 = get_epsilon(5000)
print(f"Epsilon at step 10: {res1}")
print(f"Epsilon at step 3000: {res2}")
print(f"Epsilon at step 5000: {res3}")


Epsilon at step 10: 0.996999979019165
Epsilon at step 3000: 0.10000000149011612
Epsilon at step 5000: 0.10000000149011612


16.

In [33]:
def select_epsilon_greedy_action(key, q_values, num_timesteps):
    num_actions = len(q_values)

    epsilon = get_epsilon(num_timesteps)

    random_value = jax.random.uniform(key)
    should_explore = random_value < epsilon

    action = jax.lax.select(
        should_explore,
        select_random_action(key, num_actions),
        select_greedy_action(q_values)
    )
    return action

проверка

In [34]:
rng = hk.PRNGSequence(jax.random.PRNGKey(42))
dummy_q_values = jnp.array([0.0, 1.0])
actions_greedy = []
for i in range(5):
    action = select_epsilon_greedy_action(next(rng), dummy_q_values, 5000)
    actions_greedy.append(int(action))
actions_explore = []
for i in range(5):
    action = select_epsilon_greedy_action(next(rng), dummy_q_values, 0)
    actions_explore.append(int(action))
print(f"Жадные действия (много шагов): {actions_greedy}")
print(f"Исследовательские действия (мало шагов): {actions_explore}")

Жадные действия (много шагов): [1, 1, 1, 1, 1]
Исследовательские действия (мало шагов): [0, 1, 0, 0, 0]
